Добрый день!

Сегодня посмотрим трассировку одной функции

ДЗ выполнила **Маслова Татьяна**

## Часть 1. Карта зависимостей StPlatform

### 1.1. Слои, обнаруженные в проекте

Проект StPlatform использует доменную структуру, в котором код сгруппирован по бизнес-функциям (пакеты `api`, `submission`, `user` и т.д.). Внутри каждого пакета выделяются свои слои.

Для функции, предназначенной для сдачи домашнего задания, задействованы следующие слои:

| Слой | Класс | Пакет |
|------|-------|-------|
| **Контроллер (Web / API)** | `ApiSubmissionController` | `ru.urfu.platform.api` |
| **Сервис (Business Logic)** | `SubmissionService` | `ru.urfu.platform.submission` |
| **Сервис (пользователи)** | `UserService` | `ru.urfu.platform.user` |
| **Репозиторий (Persistence)** | `SubmissionRepository`, `AssignmentRepository`, `SubmissionAttachmentRepository` | `ru.urfu.platform.submission` |
| **Сущность (Domain / Model)** | `Submission` | `ru.urfu.platform.submission` |

### 1.2. Трассировка функции «Сдать домашку»

На основе кода ApiSubmissionController.java и SubmissionService.java:

ApiSubmissionController.submit() — принимает POST-запрос /api/submissions с телом NewSubmissionRequest (ID задания, ссылки, комментарий) и токеном пользователя.

UserService.byId(principal.getId()) — контроллер запрашивает данные студента по ID из токена авторизации.

SubmissionService.submit(student, assignmentId, links, comment, null) — контроллер делегирует создание сдачи сервису.

AssignmentRepository.findById(assignmentId) — внутри сервиса проверяется, существует ли задание и опубликовано ли оно.

SubmissionRepository.save(new Submission(...)) — сервис сохраняет новую сущность сдачи в базу данных.

SubmissionAttachmentRepository.save(...) — если приложен файл, сохраняется вложение.

ApplicationEventPublisher.publishEvent(...) — публикуется событие для ИИ-проверки.

Все вызовы идут строго «сверху вниз» (Контроллер → Сервис → Репозиторий). Обратных вызовов нет.
В методе `submit()` контроллер сам вызывает `UserService`, а потом `SubmissionService`, то есть выступает оркестратором. Я бы перенесла получение пользователя внутрь `SubmissionService`, чтобы контроллер вызывал ровно один метод сервиса (`submissionService.processSubmission(studentId, request)`), чтобы изменение способа хранения данных или правил проверки (дедлайны, права) не затрагивало код обработки HTTP-запросов.


In [ ]:
// Было:
User student = users.byId(principal.getId());
Submission submission = submissions.submit(student, ...);

// Стало бы:
Submission submission = submissions.submitForUser(principal.getId(), request);

Тогда контроллер отвечает только за приём HTTP-запроса и возврат DTO, а вся бизнес-логика (найти студента, проверить дедлайн, сохранить) сосредоточена в сервисе.

Граница нужна для того, чтобы изменение способа хранения данных или правил проверки (дедлайны, права) не затрагивало код обработки HTTP-запросов, а контроллер оставался тонким и тестируемым изолированно.

## Часть 2. Трассировка PetClinic через агента

Инструмент: AI-ассистент Perplexity (веб-версия, бесплатная).

Промпт агенту:

«Протрассируй сценарий добавления визита в Spring PetClinic, начиная с VisitController. Выпиши цепочку вызовов в формате Файл.метод → Файл.метод. Укажи, какие слои задействованы и есть ли в этом проекте отдельный сервисный слой»

Слои, задействованные в PetClinic (сценарий добавления визита):

| Слой | Классы / компоненты | Роль |
|------|---------------------|------|
| **Web / Presentation** | `VisitController`, Thymeleaf-форма `pets/createOrUpdateVisitForm` | Принимает запрос, валидирует `Visit`, формирует redirect |
| **Domain / Model** | `Owner`, `Pet`, `Visit` | Инкапсулируют связь владельца, питомца и визита |
| **Persistence / Repository** | `OwnerRepository` | Загружает и сохраняет агрегат `Owner` через Spring Data JPA |

Цепочка вызовов (сценарий «добавление визита»)
VisitController.processNewVisitForm() — точка входа после привязки формы и валидации (POST-запрос).

VisitController.loadPetWithVisit() — подготовительный метод, который загружает владельца, находит питомца и создаёт пустой объект Visit:
OwnerRepository.findById() → Owner.getPet() → Pet.addVisit().

Owner.addVisit(petId, visit) — связывает визит с питомцем.

OwnerRepository.save(owner) — сохраняет весь агрегат Owner (вместе с новым визитом) в базу данных.

При ошибке валидации: VisitController.processNewVisitForm() возвращает представление pets/createOrUpdateVisitForm, сохранение не выполняется.

При успехе: VisitController.processNewVisitForm() → RedirectAttributes.addFlashAttribute() → редирект на /owners/{ownerId} с сообщением «Your visit has been booked».





При этом, все стрелки идут «сверху вниз» от контроллера к репозиторию, а обратных вызовов нет.
В основном репозитории spring-projects/spring-petclinic отдельного сервиса для этого сценария нет. VisitController напрямую инжектирует OwnerRepository и вызывает его методы. Между контроллером и репозиторием нет VisitService или ClinicService.

Агент также отметил, что в других вариантах PetClinic (например, spring-petclinic-rest) сервисный слой присутствует (ClinicServiceImpl.saveVisit() вызывает VisitRepository.save()), но это другой проект, а не указанный spring-projects/spring-petclinic.

В PetClinic граница между контроллером и доступом к данным размыта.Контроллер знает о методе findById и о том, как сохранять сущности. Можно провести границу между VisitController и репозиторием, добавив тонкий VisitService, который инкапсулировал бы логику загрузки питомца и сохранения визита. Тогда контроллер вызывал бы один метод сервиса, а не два метода репозитория. Граница защитит контроллер от изменений в модели данных и для переиспользования логики добавления визита в других сценариях.

В StPlatform архитектура более строгая, в которой есть сервисный слой, содержащий всю бизнес-логику (проверка ролей, дедлайнов, валидация ссылок), а контроллер остаётся тонким. В PetClinic архитектура упрощена, где контроллер напрямую работает с репозиторием, что допустимо для небольших операций, но размывает границу между представлением и доступом к данным.

Обе трассировки показывают, что направление зависимостей идёт «сверху вниз», а необходимость сервисного слоя определяется сложностью бизнес-операции.